# New Section

In [ ]:
# 📦 Cell 1: Install & import (run once)
!pip install -q gradio reportlab
import html, re, os, json, csv, traceback, pandas as pd, gradio as gr
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer
print("Ready ✅")


Ready ✅


# file handling

In [6]:
# 🎓 Cell 2: Student Directory Dashboard — ONE CSV with all students, then search by name or surname

# ---------- Reading the single CSV ----------
def _label_value(cells, label):
    """Reads 'Course: BSCS' or 'Course,BSCS' style lines."""
    first = cells[0].strip()
    if first.lower().rstrip(":").strip() == label and len(cells) > 1:
        return cells[1].strip()
    if first.lower().startswith(label) and ":" in first:
        return first.split(":", 1)[1].strip()
    return None

_LABELS = ("course", "year", "name")

def _is_meta(cells):
    """True for lines like 'Julkiffli M. Abo, Course: BSCS, Year: 1st Year' (labels can share one row with the name)."""
    return any(":" in c and c.split(":", 1)[0].strip().lower() in _LABELS for c in cells)

def _meta_info(meta):
    name = course = year = ""
    for cells in meta:
        lab0 = cells[0].lower().rstrip(":").strip()
        if lab0 in _LABELS and len(cells) > 1:                  # 'Course,BSCS'
            pairs, rest = [(lab0, cells[1].strip())], cells[2:]
        else:
            pairs, rest = [], cells
        plain = []
        for c in rest:
            if ":" in c and c.split(":", 1)[0].strip().lower() in _LABELS:   # 'Course: BSCS'
                l, v = c.split(":", 1); pairs.append((l.strip().lower(), v.strip()))
            else:
                plain.append(c)
        for l, v in pairs:
            if l == "course": course = v
            elif l == "year": year = v
            elif l == "name": name = v
        if plain and not name:
            name = " ".join(plain)
    return name, course, year

def _num(x):
    try:
        return float(str(x).strip())
    except ValueError:
        return None

def parse_students(file, def_course="", def_year=""):
    """Accepts ONE csv that contains many students. Two layouts work (even mixed):
       A) one table with a NAME column (+ optional COURSE / YEAR columns), one row per subject
       B) stacked blocks: a name line, then the CODE/DESCRIPTION/UNITS/TEACHER/GRADE header, then that student's subjects"""
    path = file if isinstance(file, str) else file.name
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(path, newline="", encoding=enc) as f:
                rows = list(csv.reader(f))
            break
        except UnicodeDecodeError:
            continue

    aliases = {"DISCRIPTION": "DESCRIPTION", "DESCRIPTI": "DESCRIPTION", "SUBJECT": "DESCRIPTION",
               "INSTRUCTOR": "TEACHER", "PROFESSOR": "TEACHER",
               "STUDENT": "NAME", "STUDENT NAME": "NAME", "FULL NAME": "NAME",
               "PROGRAM": "COURSE", "YEAR LEVEL": "YEAR"}
    students, meta, header, blk, last = {}, [], None, ("", "", ""), {}
    for row in rows:
        cells = [c.strip() for c in row]
        nonempty = [c for c in cells if c]
        up = [c.upper() for c in cells]
        if "CODE" in up and "GRADE" in up:                      # a header row
            header = [aliases.get(c, c) for c in up]
            miss = [c for c in ("DESCRIPTION", "UNITS") if c not in header]
            if miss:
                raise ValueError(f"missing column(s) {miss}")
            blk, meta, last = _meta_info(meta), [], {}
            continue
        if not nonempty:
            continue
        if header is not None and len(nonempty) >= 3 and not _is_meta(nonempty):   # a subject row
            get = lambda col: cells[header.index(col)] if col in header and header.index(col) < len(cells) else ""
            for col in ("NAME", "COURSE", "YEAR"):              # blank cell = same as the row above (merged cells)
                if get(col): last[col] = get(col)
            name   = last.get("NAME")   or blk[0] or "Unnamed student"
            course = last.get("COURSE") or blk[1]
            year   = last.get("YEAR")   or blk[2]
            units, grade = _num(get("UNITS")), _num(get("GRADE"))
            if units is None or grade is None:
                continue
            st = students.setdefault(name, {"course": "", "year": "", "rows": []})
            st["course"] = st["course"] or course
            st["year"]   = st["year"] or year
            st["rows"].append({"CODE": get("CODE"), "DESCRIPTION": get("DESCRIPTION"), "UNITS": units,
                               "TEACHER": get("TEACHER"), "GRADE": grade})
        else:                                                   # a text line (name / course / year) -> next block
            header = None
            meta.append(nonempty)
    if not students:
        raise ValueError("no student grades found. Make sure the file has CODE, DESCRIPTION, UNITS, TEACHER and GRADE columns")
    for st in students.values():
        st["course"] = st["course"] or def_course or "—"
        st["year"]   = st["year"] or def_year
    return students

def stats(df):
    units = df["UNITS"].sum()
    return {"subjects": len(df), "units": units,
            "best": df["GRADE"].min(), "lowest": df["GRADE"].max(),
            "gwa": (df["GRADE"] * df["UNITS"]).sum() / units}   # unit-weighted GWA

def grade_color(g):          # 1.00 = best
    if g <= 1.5: return "#22c55e"
    if g <= 2.0: return "#38bdf8"
    if g <= 2.5: return "#f59e0b"
    return "#ef4444"

def standing(g):
    if g <= 1.5: return "Excellent"
    if g <= 2.0: return "Very Good"
    if g <= 2.5: return "Good"
    if g <= 3.0: return "Passing"
    return "Needs Improvement"

def initials(name):
    parts = [p for p in name.split() if p]
    return ("".join(p[0] for p in parts[:2]) or "?").upper()

# ---------- On-screen report (every colour is set explicitly so dark/light mode can't break it) ----------
def render_html(df, name, course, year):
    s = stats(df)
    n  = html.escape(name.strip()   or "Student Name")
    c  = html.escape(course.strip() or "Course")
    y  = html.escape(year)
    rows = ""
    for i, r in enumerate(df.itertuples()):
        col = grade_color(r.GRADE)
        rows += (f"<tr class='sr-row {'alt' if i%2 else ''}'>"
                 f"<td class='code'>{html.escape(str(r.CODE))}</td>"
                 f"<td>{html.escape(str(r.DESCRIPTION))}</td>"
                 f"<td class='ctr'>{r.UNITS:g}</td>"
                 f"<td class='muted'>{html.escape(str(r.TEACHER))}</td>"
                 f"<td class='rt'><span class='pill' style='background:{col}22;color:{col};border:1px solid {col}'>{r.GRADE:.2f}</span></td></tr>")
    gcol = grade_color(s["gwa"])
    return f"""
    <div class='sr'>
      <div class='sr-hero'>
        <div class='avatar'>{initials(name)}</div>
        <div class='who'>
          <div class='tag'>STUDENT ACADEMIC PERFORMANCE REPORT</div>
          <div class='nm'>{n}</div>
          <div class='chips'><span class='chip'>🎓 {c}</span><span class='chip'>📅 {y}</span></div>
        </div>
        <div class='gwa-big' style='border-color:{gcol}'>
          <div class='lbl'>GWA</div><div class='val' style='color:{gcol}'>{s['gwa']:.2f}</div>
          <div class='sub' style='color:{gcol}'>{standing(s['gwa'])}</div>
        </div>
      </div>
      <div class='cards'>
        <div class='card'><div class='lbl'>SUBJECTS</div><div class='val'>{s['subjects']}</div></div>
        <div class='card'><div class='lbl'>UNITS EARNED</div><div class='val'>{s['units']:.1f}</div></div>
        <div class='card'><div class='lbl'>BEST GRADE</div><div class='val' style='color:#22c55e'>{s['best']:.2f}</div></div>
        <div class='card'><div class='lbl'>LOWEST GRADE</div><div class='val' style='color:#f87171'>{s['lowest']:.2f}</div></div>
      </div>
      <div class='sr-title'>Grade Report</div>
      <div class='tblwrap'><table class='sr-tbl'>
        <thead><tr><th>CODE</th><th>DESCRIPTION</th><th class='ctr'>UNITS</th><th>TEACHER</th><th class='rt'>GRADE</th></tr></thead>
        <tbody>{rows}</tbody></table></div>
    </div>"""

CSS = """

#hdr {border-bottom:1px solid #6366f1; padding-bottom:6px}
#upd {background:linear-gradient(90deg,#6366f1,#06b6d4) !important; color:white !important; border:none !important}
#pdf {background:#f59e0b !important; color:#1f1300 !important; border:none !important}

.sr, .sr * {box-sizing:border-box; font-family:'Segoe UI',Arial,sans-serif}
.sr {background:#0b1020; border-radius:16px; padding:22px; color:#e5e7eb !important}
.sr div, .sr span, .sr td, .sr th {color:#e5e7eb !important; margin:0}
.sr .muted {color:#94a3b8 !important}
.sr-hero {display:flex; align-items:center; gap:18px; padding:20px; border-radius:14px;
          background:linear-gradient(135deg,#312e81,#0e7490)}
.sr .avatar {width:88px; height:88px; border-radius:50%; background:#ffffff22; border:2px solid #ffffff88;
             display:flex; align-items:center; justify-content:center; font-size:32px; font-weight:700; color:#fff !important}
.sr .who {flex:1}
.sr .tag {font-size:14px; letter-spacing:2px; color:#c7d2fe !important}
.sr .nm {font-size:38px; font-weight:800; color:#ffffff !important; margin:2px 0 8px !important}
.sr .chip {display:inline-block; background:#ffffff22; padding:4px 12px; border-radius:20px; margin-right:8px; font-size:16px; color:#fff !important}
.sr .gwa-big {text-align:center; background:#0b1020cc; border:2px solid; border-radius:14px; padding:12px 22px}
.sr .gwa-big .lbl {font-size:14px; letter-spacing:2px; color:#94a3b8 !important}
.sr .gwa-big .val {font-size:48px; font-weight:800}
.sr .gwa-big .sub {font-size:15px; font-weight:600}
.sr .cards {display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin:14px 0}
.sr .card {background:#151b30; border:1px solid #232b45; border-radius:12px; padding:14px; text-align:center}
.sr .card .lbl {font-size:14px; letter-spacing:1.5px; color:#94a3b8 !important}
.sr .card .val {font-size:32px; font-weight:700; color:#ffffff}
.sr .sr-title {font-size:19px; font-weight:700; color:#a5b4fc !important; margin:8px 0 8px !important}
.sr .tblwrap {overflow-x:auto; border-radius:12px; border:1px solid #232b45}
.sr table {width:100%; border-collapse:collapse; background:transparent !important; margin:0 !important}
.sr th {background:#1e2547 !important; color:#a5b4fc !important; font-size:15px; letter-spacing:1px; text-align:left; padding:12px 10px; border:none !important}
.sr td {padding:11px 10px; font-size:16px; border:none !important; border-top:1px solid #1c2340 !important; background:#121830 !important}
.sr tr.alt td {background:#0f1428 !important}
.sr .code {font-weight:700; color:#ffffff !important}
.sr .ctr {text-align:center !important} .sr .rt {text-align:right !important}
.sr .pill {display:inline-block; min-width:54px; text-align:center; padding:3px 10px; border-radius:20px; font-weight:700}
@media (max-width:700px){ .sr-hero{flex-wrap:wrap} .sr .cards{grid-template-columns:repeat(2,1fr)} }

/* wider, centred page */
.gradio-container {max-width: 1500px !important; width: 100% !important; margin-left: auto !important;
                   margin-right: auto !important; padding: 0 28px !important; font-size: 17px !important}
#hdr {text-align: center; font-size: 30px !important}
#hdr h2 {font-size: 32px !important; margin: 18px 0 !important}
button[role="tab"] {font-size: 18px !important; padding: 12px 22px !important}
.gradio-container label span, .gradio-container .prose p {font-size: 16px !important}
.gradio-container input, .gradio-container textarea, .gradio-container button.lg {font-size: 16px !important}
.sr {padding: 34px !important}
.sr .tblwrap, .sr .cards {width: 100%}
"""

# ---------- PDF ----------
def make_pdf(df, name, course, year, path="Student_Report.pdf", story_only=False):
    s = stats(df)
    navy, indigo, teal = colors.HexColor("#0b1020"), colors.HexColor("#312e81"), colors.HexColor("#0e7490")
    doc = SimpleDocTemplate(path, pagesize=landscape(A4), leftMargin=30, rightMargin=30, topMargin=30, bottomMargin=30)
    W = landscape(A4)[0] - 60
    tag  = ParagraphStyle("tag", fontName="Helvetica", fontSize=9, textColor=colors.HexColor("#c7d2fe"))
    nm   = ParagraphStyle("nm",  fontName="Helvetica-Bold", fontSize=24, textColor=colors.white, leading=30)
    sub  = ParagraphStyle("sub", fontName="Helvetica", fontSize=12, textColor=colors.white)
    cell = ParagraphStyle("c",   fontName="Helvetica", fontSize=9)
    gw   = ParagraphStyle("gw",  fontName="Helvetica-Bold", fontSize=26, textColor=colors.white, alignment=1, leading=30)
    gl   = ParagraphStyle("gl",  fontName="Helvetica", fontSize=8, textColor=colors.HexColor("#c7d2fe"), alignment=1)

    left = [Paragraph("STUDENT ACADEMIC PERFORMANCE REPORT", tag),
            Paragraph(html.escape(name.strip() or "Student Name"), nm),
            Paragraph(f"{html.escape(course)}  &nbsp;|&nbsp;  {html.escape(year)}", sub)]
    right = [Paragraph("GWA", gl), Paragraph(f"{s['gwa']:.2f}", gw), Paragraph(standing(s['gwa']), gl)]
    head = Table([[left, right]], colWidths=[W*.78, W*.22])
    head.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,-1), indigo), ("BACKGROUND", (1,0), (1,0), teal),
                              ("VALIGN", (0,0), (-1,-1), "MIDDLE"), ("LEFTPADDING", (0,0), (-1,-1), 16),
                              ("TOPPADDING", (0,0), (-1,-1), 14), ("BOTTOMPADDING", (0,0), (-1,-1), 14)]))

    data = [["CODE", "DESCRIPTION", "UNITS", "TEACHER", "GRADE"]]
    for r in df.itertuples():
        data.append([str(r.CODE), Paragraph(html.escape(str(r.DESCRIPTION)), cell), f"{r.UNITS:g}",
                     Paragraph(html.escape(str(r.TEACHER)), cell), f"{r.GRADE:.2f}"])
    tbl = Table(data, colWidths=[W*.12, W*.38, W*.08, W*.28, W*.14], repeatRows=1)
    style = [("BACKGROUND", (0,0), (-1,0), indigo), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
             ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"), ("FONTSIZE", (0,0), (-1,-1), 9),
             ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.HexColor("#f8fafc"), colors.HexColor("#eef2ff")]),
             ("ALIGN", (2,0), (2,-1), "CENTER"), ("ALIGN", (4,0), (4,-1), "RIGHT"),
             ("VALIGN", (0,0), (-1,-1), "MIDDLE"), ("FONTNAME", (4,1), (4,-1), "Helvetica-Bold"),
             ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6)]
    for i, r in enumerate(df.itertuples(), start=1):
        style.append(("TEXTCOLOR", (4,i), (4,i), colors.HexColor(grade_color(r.GRADE))))
    tbl.setStyle(TableStyle(style))

    lab = ParagraphStyle("l", fontName="Helvetica", fontSize=8, alignment=1, textColor=colors.HexColor("#64748b"))
    val = ParagraphStyle("v", alignment=1, leading=22)
    def sbox(label, v, col="#312e81"):
        return [Paragraph(label, lab), Paragraph(f"<font size=16 color='{col}'><b>{v}</b></font>", val)]
    b = [sbox("TOTAL SUBJECTS", s["subjects"]), sbox("UNITS EARNED", f"{s['units']:.1f}"),
         sbox("BEST GRADE", f"{s['best']:.2f}", "#16a34a"), sbox("LOWEST GRADE", f"{s['lowest']:.2f}", "#dc2626")]
    summ = Table([[x[0] for x in b], [x[1] for x in b]], colWidths=[W/4]*4)
    summ.setStyle(TableStyle([("BOX", (0,0), (-1,-1), .5, colors.lightgrey), ("INNERGRID", (0,0), (-1,-1), .25, colors.lightgrey),
                              ("BACKGROUND", (0,0), (-1,-1), colors.HexColor("#f8fafc")),
                              ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6)]))
    story = [head, Spacer(1,14), summ, Spacer(1,14), tbl]
    if story_only:
        return story
    doc.build(story)
    return path

def ordered_names(sort="Name (A–Z)"):
    names = sorted(DB, key=str.lower)
    if str(sort).startswith("Best"):
        names.sort(key=lambda n: stats(student_df(n))["gwa"])
    return names

def make_all_pdf(sort="Name (A–Z)", path="All_Students_Report.pdf"):
    """One PDF: a ranking page first, then each student's full report on its own page."""
    from reportlab.platypus import PageBreak
    names = ordered_names(sort)
    indigo = colors.HexColor("#312e81")
    W = landscape(A4)[0] - 60
    ttl = ParagraphStyle("ttl", fontName="Helvetica-Bold", fontSize=18, textColor=indigo, leading=24)
    sub = ParagraphStyle("sb", fontName="Helvetica", fontSize=10, textColor=colors.HexColor("#64748b"))
    cell = ParagraphStyle("cl", fontName="Helvetica", fontSize=9)
    data = [["#", "NAME", "COURSE", "YEAR", "SUBJECTS", "UNITS", "GWA", "EVALUATION"]]
    for i, n in enumerate(names, 1):
        s = stats(student_df(n))
        data.append([i, Paragraph(html.escape(n), cell), Paragraph(html.escape(str(DB[n]["course"])), cell),
                     DB[n]["year"], s["subjects"], f"{s['units']:.1f}", f"{s['gwa']:.2f}", standing(s["gwa"])])
    tbl = Table(data, colWidths=[W*.05, W*.27, W*.13, W*.12, W*.10, W*.09, W*.09, W*.15], repeatRows=1)
    style = [("BACKGROUND", (0,0), (-1,0), indigo), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
             ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"), ("FONTSIZE", (0,0), (-1,-1), 9),
             ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.HexColor("#f8fafc"), colors.HexColor("#eef2ff")]),
             ("ALIGN", (4,0), (-1,-1), "CENTER"), ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
             ("FONTNAME", (6,1), (6,-1), "Helvetica-Bold"),
             ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6)]
    for i, n in enumerate(names, 1):
        gc = colors.HexColor(grade_color(stats(student_df(n))["gwa"]))
        style += [("TEXTCOLOR", (6,i), (7,i), gc), ("FONTNAME", (7,i), (7,i), "Helvetica-Bold")]
    tbl.setStyle(TableStyle(style))
    story = [Paragraph("All Students — Academic Performance Summary", ttl),
             Paragraph(f"{len(names)} student(s)", sub), Spacer(1, 12), tbl]
    for n in names:
        d = DB[n]
        story += [PageBreak()] + make_pdf(student_df(n), n, d["course"], d["year"], story_only=True)
    SimpleDocTemplate(path, pagesize=landscape(A4), leftMargin=30, rightMargin=30, topMargin=30, bottomMargin=30,
                      title="All Students Report").build(story)
    return path



# ---------- Student directory (saved to a file so it survives page refreshes) ----------
DB_FILE = os.path.join("/content" if os.path.isdir("/content") else ".", "student_profiles.json")
DB = json.load(open(DB_FILE)) if os.path.exists(DB_FILE) else {}

def save_db():
    json.dump(DB, open(DB_FILE, "w"), indent=1)

def norm(s):
    return re.sub(r"[^a-z0-9 ]", " ", str(s).lower())

def find(q):
    """Every word typed must appear in the name, so 'abo', 'julkiffli' or 'julk abo' all work."""
    toks = norm(q).split()
    names = sorted(DB, key=str.lower)
    return [n for n in names if all(t in norm(n) for t in toks)] if toks else names

def student_df(name):
    return pd.DataFrame(DB[name]["rows"])

def table_rows():
    out = []
    for n in sorted(DB, key=str.lower):
        d = student_df(n); s = stats(d)
        out.append([n, DB[n]["course"], DB[n]["year"], s["subjects"], round(float(s["units"]), 1), round(float(s["gwa"]), 2)])
    return out


# ---------- GUI actions ----------
HINT = "<div style='padding:24px;text-align:center;color:#94a3b8;font-family:Arial'>{}</div>"

SORT_CHOICES = ["Name (A–Z)", "Best GWA first"]

def render_all(sort="Name (A–Z)"):
    """A summary table of everyone, followed by every student's full report."""
    if not DB:
        return HINT.format("No students yet — load the CSV file in the first tab.")
    names = sorted(DB, key=str.lower)
    if str(sort).startswith("Best"):
        names.sort(key=lambda n: stats(student_df(n))["gwa"])
    rows = ""
    for i, n in enumerate(names):
        s = stats(student_df(n)); col = grade_color(s["gwa"])
        rows += (f"<tr class='sr-row {'alt' if i % 2 else ''}'><td class='code'>{i + 1}</td>"
                 f"<td>{html.escape(n)}</td><td class='muted'>{html.escape(str(DB[n]['course']))}</td>"
                 f"<td class='muted'>{html.escape(str(DB[n]['year']))}</td><td class='ctr'>{s['subjects']}</td>"
                 f"<td class='ctr'>{s['units']:.1f}</td>"
                 f"<td class='ctr'><span class='pill' style='background:{col}22;color:{col};border:1px solid {col}'>{s['gwa']:.2f}</span></td>"
                 f"<td class='rt' style='color:{col} !important;font-weight:700'>{standing(s['gwa'])}</td></tr>")
    summary = (f"<div class='sr'><div class='sr-title'>All Students — {len(names)} total</div><div class='tblwrap'><table class='sr-tbl'>"
               "<thead><tr><th>#</th><th>NAME</th><th>COURSE</th><th>YEAR</th><th class='ctr'>SUBJECTS</th>"
               "<th class='ctr'>UNITS</th><th class='ctr'>GWA</th><th class='rt'>EVALUATION</th></tr></thead>"
               f"<tbody>{rows}</tbody></table></div></div>")
    reports = "".join("<div style='height:18px'></div>" + render_html(student_df(n), n, DB[n]["course"], DB[n]["year"])
                      for n in names)
    return summary + reports

def ui_load(file, def_course, def_year, sort):
    if file is None:
        raise gr.Error("Please choose the CSV file first.")
    try:
        students = parse_students(file, def_course, def_year)
        DB.clear(); DB.update(students); save_db()          # the file replaces the whole directory
        names = sorted(DB, key=str.lower)
        msg = f"✅ Loaded **{len(names)}** student(s): " + ", ".join(names) + "\n\nGo to the **Search Student** tab."
        return msg, table_rows(), gr.Dropdown(choices=names, value=None), render_all(sort)
    except Exception as e:
        traceback.print_exc(); raise gr.Error(f"{type(e).__name__}: {e}")

def ui_search(q):
    matches = find(q)
    if not DB:
        info = "No students yet — load the CSV file in the first tab."
    elif not q.strip():
        info = f"{len(DB)} student(s) in the directory — type a name or surname."
    elif not matches:
        info = f"❌ No student found for “{q}”."
    else:
        info = f"✅ {len(matches)} match(es)" + (" — choose one below." if len(matches) > 1 else ".")
    first = matches[0] if (matches and q.strip()) else None
    return gr.Dropdown(choices=matches, value=first), info

def ui_show(name):
    if not name or name not in DB:
        return HINT.format("Type a name or surname above to see a student’s grades and course.")
    d = DB[name]
    return render_html(student_df(name), name, d["course"], d["year"])

def ui_pdf(name):
    if not name or name not in DB:
        raise gr.Error("Search and select a student first.")
    try:
        d = DB[name]
        path = f"{re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')}_report.pdf"
        return gr.File(value=make_pdf(student_df(name), name, d["course"], d["year"], path), visible=True)
    except Exception as e:
        traceback.print_exc(); raise gr.Error(f"{type(e).__name__}: {e}")

def ui_pdf_all(sort):
    if not DB:
        raise gr.Error("Load the CSV file first.")
    try:
        return gr.File(value=make_all_pdf(sort), visible=True)
    except Exception as e:
        traceback.print_exc(); raise gr.Error(f"{type(e).__name__}: {e}")

# ---------- GUI layout ----------
DIR_HEADERS = ["Name", "Course", "Year", "Subjects", "Units", "GWA"]
import inspect, warnings
warnings.filterwarnings("ignore")                                   # hide Gradio's deprecation warnings
CSS_IN_LAUNCH = "css" in inspect.signature(gr.Blocks.launch).parameters   # Gradio 6+ takes css in launch()
with gr.Blocks(title="Student Directory Dashboard", **({} if CSS_IN_LAUNCH else {"css": CSS})) as demo:
    gr.Markdown("## 🎓 Student Directory Dashboard", elem_id="hdr")
    with gr.Tabs():
        with gr.Tab("📂 Upload File"):
            gr.Markdown("Upload **one CSV** that contains all the students. Either layout works: one table with a `NAME` column "
                        "(plus `COURSE` / `YEAR` columns if you have them), **or** each student's block one after another "
                        "(name line, then the `CODE, DESCRIPTION, UNITS, TEACHER, GRADE` header, then their subjects). "
                        "The defaults below are used when a student has no course or year in the file.")
            with gr.Row():
                csv_file = gr.File(label="Upload CSV (all students)", file_count="single", file_types=[".csv"], type="filepath")
                with gr.Column():
                    def_course = gr.Textbox(label="Default course", placeholder="e.g. BSCS")
                    def_year = gr.Dropdown(["1st Year", "2nd Year", "3rd Year", "4th Year"], value="1st Year", label="Default year")
                    btn_load = gr.Button("Load File", elem_id="upd")
            status = gr.Markdown()
            directory = gr.Dataframe(value=table_rows(), headers=DIR_HEADERS, interactive=False, label="Directory")
        with gr.Tab("🔎 Search Student"):
            q = gr.Textbox(label="Search by name or surname", placeholder="type e.g. “abo” or “julkiffli”…")
            info = gr.Markdown()
            pick = gr.Dropdown(choices=sorted(DB, key=str.lower), value=None, label="Matching students")
            with gr.Row():
                btn_pdf = gr.Button("Save to PDF", elem_id="pdf")
            pdf_out = gr.File(label="Your PDF is ready — click to download", visible=False)
            report = gr.HTML(HINT.format("Type a name or surname above to see a student’s grades and course."))

        with gr.Tab("📋 All Students"):
            with gr.Row():
                sort_dd = gr.Dropdown(SORT_CHOICES, value=SORT_CHOICES[0], label="Sort by")
                btn_refresh = gr.Button("Refresh", elem_id="upd")
                btn_pdf_all = gr.Button("Save All to PDF", elem_id="pdf")
            pdf_all_out = gr.File(label="Your PDF is ready — click to download", visible=False)
            all_report = gr.HTML(render_all())

    btn_load.click(ui_load, [csv_file, def_course, def_year, sort_dd], [status, directory, pick, all_report])
    sort_dd.change(render_all, sort_dd, all_report)
    btn_refresh.click(render_all, sort_dd, all_report)
    btn_pdf_all.click(ui_pdf_all, sort_dd, pdf_all_out)
    q.change(ui_search, q, [pick, info])
    pick.change(ui_show, pick, report)
    btn_pdf.click(ui_pdf, pick, pdf_out)

# share=True is required on Colab anyway; quiet=True hides the log messages
demo.launch(share=True, quiet=True, show_error=True, **({"css": CSS} if CSS_IN_LAUNCH else {}))


* Running on public URL: https://b174e72df5bb8e9d40.gradio.live
